# Final Project Starter Notebook
## Dynamic Asset Pricing with the Weekly Economic Index

This notebook is a **starter file**. It is designed to help you organize your work, but you must complete the analysis, write up your explanations, and make sure every answer is your own.

**Important reminders**

- Use the **maximum available sample** of the Weekly Economic Index (WEI) from FRED.
- WEI is reported in **percent**, so convert it to **decimals** by dividing by 100.
- Use **20 bins**.
- These are **weekly data**, so the discount factor is

\[
\beta = 0.97^{1/52}.
\]

Fill in the missing pieces marked `TODO`.

In [ ]:

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.linalg import eig, solve

# Optional: if needed
# import scipy.linalg as la

## 0. Download the Weekly Economic Index from FRED

Use the full available sample. Convert the series from percent to decimal form.

In [ ]:

# TODO: If pandas_datareader is not installed, install it in your environment.
# %pip install pandas_datareader

from pandas_datareader import data as web

series_name = "WEI"
start_date = "2020-01-01"   # TODO: you may adjust if needed, but use the full available sample
end_date = None             # None means up to the latest available date

wei = web.DataReader(series_name, "fred", start_date, end_date)
wei = wei.dropna().rename(columns={series_name: "WEI_percent"})

# Convert from percent to decimal
wei["WEI"] = wei["WEI_percent"] / 100.0

wei.head(), wei.tail(), wei.shape

## 1. Plot the WEI and construct a histogram with 20 bins

Interpret the bins as states of the economy.

In [ ]:

fig, ax = plt.subplots(figsize=(10, 4))
wei["WEI"].plot(ax=ax)
ax.set_title("Weekly Economic Index (decimal form)")
ax.set_ylabel("WEI")
ax.set_xlabel("Date")
plt.show()

# Histogram with 20 bins
fig, ax = plt.subplots(figsize=(8, 4))
counts, bin_edges, _ = ax.hist(wei["WEI"], bins=20, edgecolor="black")
ax.set_title("Histogram of WEI with 20 bins")
ax.set_xlabel("WEI")
ax.set_ylabel("Frequency")
plt.show()

print("Counts by bin:")
print(counts.astype(int))
print("Bin edges:")
print(bin_edges)

## 2. Classify each weekly observation into one of the 20 states

Use the **same bin boundaries** as in the histogram.

State 1 should be the **lowest** WEI bin.

In [ ]:

# Use pd.cut with the histogram bin edges
wei["state"] = pd.cut(
    wei["WEI"],
    bins=bin_edges,
    labels=False,
    include_lowest=True,
    right=True
)

# pd.cut labels bins 0,...,19; shift to 1,...,20
wei["state"] = wei["state"] + 1

# Check for missing classifications
print("Missing state assignments:", wei["state"].isna().sum())

state_counts = wei["state"].value_counts().sort_index()
print(state_counts)

wei.head()

## 3. Estimate the 20×20 Markov transition matrix

Let

\[
P_{ij} = \Pr(s_{t+1}=j \mid s_t=i).
\]

Then compute the ergodic distribution.

In [ ]:

# Build transition counts
states = wei["state"].astype(int).to_numpy()
n_states = 20

N = np.zeros((n_states, n_states), dtype=int)

for t in range(len(states) - 1):
    i = states[t] - 1
    j = states[t + 1] - 1
    N[i, j] += 1

N_i = N.sum(axis=1, keepdims=True)

# Avoid division by zero, though every state should ideally appear
P = np.divide(N, N_i, where=(N_i != 0))

# Check row sums
row_sums = P.sum(axis=1)
print("Row sums:")
print(row_sums)

P_df = pd.DataFrame(P, index=np.arange(1, 21), columns=np.arange(1, 21))
P_df.round(3).head()

In [ ]:

# Method 1: left eigenvector of P associated with eigenvalue 1
evals, evecs = eig(P.T)
idx = np.argmin(np.abs(evals - 1))
pi = np.real(evecs[:, idx])
pi = pi / pi.sum()

# Make sure it is nonnegative if sign flipped
if np.any(pi < 0):
    pi = -pi
    pi = pi / pi.sum()

print("Ergodic distribution:")
print(pi)
print("Sum:", pi.sum())
print("Long-run probability of Covid state (state 1):", pi[0])

## 4. Construct consumption

For each state, compute the **sample mean** of WEI within that state, not the midpoint of the bin.

Then define

\[
c_i = e^{x_i}.
\]

In [ ]:

state_means = wei.groupby("state")["WEI"].mean().sort_index()

# Make sure all 20 states are present
state_means = state_means.reindex(np.arange(1, 21))

x = state_means.to_numpy()
c = np.exp(x)

consumption_df = pd.DataFrame({
    "state": np.arange(1, 21),
    "mean_WEI": x,
    "consumption": c
})

consumption_df

## 5. Asset pricing kernel

Use CRRA utility

\[
u(c)=\frac{c^{1-\gamma}-1}{1-\gamma}, \qquad u'(c)=c^{-\gamma}
\]

with

\[
\gamma = 2, \qquad \beta = 0.97^{1/52}.
\]

In [ ]:

gamma = 2
beta = 0.97 ** (1/52)

u_prime = c ** (-gamma)

# M_ij = beta * u'(c_j) / u'(c_i)
M = beta * (u_prime[np.newaxis, :] / u_prime[:, np.newaxis])

M_df = pd.DataFrame(M, index=np.arange(1, 21), columns=np.arange(1, 21))
M_df.round(4).head()

In [ ]:

print("Diagonal entries of M:")
print(np.diag(M))

print("\nAre diagonal entries all equal (up to floating point tolerance)?")
print(np.allclose(np.diag(M), np.full(n_states, beta)))

### Written response
In a markdown cell below, explain:

1. Why the diagonal elements are all the same.
2. Why the matrix is not symmetric.
3. Why asset pricing requires both the pricing kernel and the Markov matrix.

## TODO: Write your explanation for Problem 5 here.

## 6. Yield curves

Maturities are in **quarters**:

\[
[1,2,4,8,20,40,80,120]
\]

Because this is weekly data, one quarter is **13 weeks**.

In [ ]:

maturity_quarters = np.array([1, 2, 4, 8, 20, 40, 80, 120])
maturity_weeks = 13 * maturity_quarters

A = P * M  # state-price matrix, elementwise product

# Bond prices q^{(n)} with q^{(0)} = 1
q = {0: np.ones(n_states)}

max_n = int(maturity_weeks.max())
for n in range(max_n):
    q[n + 1] = A @ q[n]

# Collect yields
yield_dict = {}
for n in maturity_weeks:
    qn = q[int(n)]
    y = -(1 / n) * np.log(qn)
    yield_dict[int(n)] = y

yield_df = pd.DataFrame(yield_dict, index=np.arange(1, 21))
yield_df.columns = [f"{n}w" for n in maturity_weeks]
yield_df.head()

In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
for i in range(n_states):
    ax.plot(maturity_quarters, yield_df.iloc[i].to_numpy(), alpha=0.8)

ax.set_title("Yield Curves by State")
ax.set_xlabel("Maturity (quarters)")
ax.set_ylabel("Yield per week")
plt.show()

### Written response
In a markdown cell below, discuss whether an inverted yield curve conveys information about next period's state.

## TODO: Write your explanation for Problem 7 here.

## 8. Price the stock market as a Lucas tree

The dividend in state \(i\) is

\[
d_i = c_i.
\]

Prices satisfy

\[
p_i = \sum_{j=1}^{20} P_{ij} M_{ij} (d_j + p_j).
\]

In vector form,

\[
(I-A)p = Ad,
\]

where \(A_{ij}=P_{ij}M_{ij}\).

In [ ]:

d = c.copy()
I = np.eye(n_states)

p = solve(I - A, A @ d)
pd_ratio = p / d

stock_df = pd.DataFrame({
    "state": np.arange(1, 21),
    "dividend": d,
    "price": p,
    "price_dividend_ratio": pd_ratio
})

stock_df

## 9. Lockdown duration

Suppose the first row of the Markov matrix is

\[
(5/6, 1/6, 0, \dots, 0).
\]

Then modify it to

\[
(p, 1-p, 0, \dots, 0)
\]

to simulate a hard 52-week lockdown.

In [ ]:

# Expected duration if staying in state 1 has probability a is 1/(1-a)
a0 = 5/6
expected_duration_original = 1 / (1 - a0)
print("Expected duration under original first row:", expected_duration_original)

# Choose p so expected duration is 52 weeks
p_lockdown = 51/52
print("p for 52-week expected duration:", p_lockdown)

In [ ]:

P_lock = P.copy()
P_lock[0, :] = 0.0
P_lock[0, 0] = p_lockdown
P_lock[0, 1] = 1 - p_lockdown

# Recompute ergodic distribution under new matrix
evals2, evecs2 = eig(P_lock.T)
idx2 = np.argmin(np.abs(evals2 - 1))
pi_lock = np.real(evecs2[:, idx2])
pi_lock = pi_lock / pi_lock.sum()
if np.any(pi_lock < 0):
    pi_lock = -pi_lock
    pi_lock = pi_lock / pi_lock.sum()

print("New long-run probability of Covid state:", pi_lock[0])

## 10. Simulate the economy for 1300 weeks under both scenarios

Start from the ergodic distribution in each scenario.

In [ ]:

rng = np.random.default_rng(12345)

def draw_initial_state(pi, rng):
    return rng.choice(np.arange(n_states), p=pi)

def simulate_states(Pmat, pi_init, T, rng):
    s = np.empty(T + 1, dtype=int)
    s[0] = draw_initial_state(pi_init, rng)
    for t in range(T):
        s[t + 1] = rng.choice(np.arange(n_states), p=Pmat[s[t], :])
    return s

T = 1300

states_status = simulate_states(P, pi, T, rng)
states_lock = simulate_states(P_lock, pi_lock, T, rng)

# Stock prices under each scenario
A_lock = P_lock * M
p_lock = solve(np.eye(n_states) - A_lock, A_lock @ d)

cons_status = c[states_status]
stock_status = p[states_status]

cons_lock = c[states_lock]
stock_lock_series = p_lock[states_lock]

In [ ]:

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(1500 * cons_status, label="1500 × consumption")
ax.plot(stock_status, label="stock market")
ax.set_title("Status Quo Scenario")
ax.set_xlabel("Week")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(1500 * cons_lock, label="1500 × consumption")
ax.plot(stock_lock_series, label="stock market")
ax.set_title("Immediate 52-Week Lockdown Scenario")
ax.set_xlabel("Week")
ax.legend()
plt.show()

## 11. Welfare and policy evaluation

This section requires written analysis. You may want to compute several summary statistics first.

Suggested components:

- long-run average consumption under each scenario
- change in the probability of the worst state
- epidemiological discussion using Sargent's SEIR code
- ethical discussion using Rawls and the veil of ignorance

In [ ]:

# Simple welfare-style summary using long-run average consumption from ergodic distributions
avg_c_status = pi @ c
avg_c_lock = pi_lock @ c

pct_loss = 100 * (avg_c_status - avg_c_lock) / avg_c_status

print("Long-run average consumption, status quo:", avg_c_status)
print("Long-run average consumption, lockdown:", avg_c_lock)
print("Percent decrease in long-run average consumption:", pct_loss)
print("Worst-state probability, status quo:", pi[0])
print("Worst-state probability, lockdown:", pi_lock[0])

## TODO: Write your response for Problem 11 here.

Be sure to discuss:

1. Economic welfare
2. Epidemiological outcomes
3. Life-years or QALYs
4. Rawls's theory of justice and the veil of ignorance

## Optional checks and cleanup
Use the cells below for any extra diagnostics, tables, or figures.

In [ ]:

# TODO: Add any extra calculations here.